# DATA266 Task 3 — CycleGAN lab runner

**A = Monet, B = Photo.** This notebook orchestrates the sibling Python scripts. It is not a standalone replacement for the project ZIP.

Run in lab Jupyter, VS Code, or Colab connected to the **lab machine's local runtime**. Keep the entire extracted repository on that machine. Git is not required there.

Read the member README before full training. No assignment results exist until you run this notebook on the permitted dataset. Smoke-test data and ratings are synthetic and must never appear as lab results.

## 1. Locate the extracted project
Edit `PROJECT_ROOT` if automatic detection fails. Paths belong to this runtime, not your laptop. For Google-hosted Colab, first copy/extract the project into its runtime. Opening the notebook alone does not copy sibling scripts.


In [ ]:
from pathlib import Path
import sys, os, json, subprocess, signal, shutil, time

# If needed replace None with Path("the extracted project folder") on THIS machine.
PROJECT_ROOT = None
if PROJECT_ROOT is None:
    candidates = [Path.cwd(), *Path.cwd().parents, Path.cwd()/"data266-lab1-pair17"]
    PROJECT_ROOT = next((p for p in candidates if (p/"task3_gan"/"Pramod_Dindukurthi"/"src"/"train.py").exists()), None)
if PROJECT_ROOT is None:
    raise RuntimeError("Set PROJECT_ROOT above to the folder containing task3_gan, then rerun this cell.")
PROJECT_ROOT = Path(PROJECT_ROOT).expanduser().resolve()
MEMBER = PROJECT_ROOT/"task3_gan"/"Pramod_Dindukurthi"
SRC = MEMBER/"src"
os.chdir(MEMBER)
print("Project found:", PROJECT_ROOT)
print("Kernel Python:", sys.executable)


## 2. Verify PyTorch, then install evaluation dependencies
Use the lab's matching CUDA PyTorch/torchvision pair. If this cell cannot import them, select the correct lab kernel or follow the lab's installation instructions. Installing requirements does not configure the NVIDIA driver.


In [ ]:
import torch, torchvision
print("PyTorch:", torch.__version__, "torchvision:", torchvision.__version__)
print("CUDA:", torch.version.cuda, "available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("VRAM GiB:", round(torch.cuda.get_device_properties(0).total_memory/2**30, 2))
else:
    print("CPU smoke tests work, but do not start full CycleGAN training until CUDA is available.")


In [ ]:
# Run once in a new environment. Restart the kernel if an already-imported package changes.
subprocess.run([sys.executable, "-m", "pip", "install", "-r", str(MEMBER/"requirements.txt")], check=True)


## 3. Runtime paths and provenance
Set `DATA_ROOT` to the parent of `monet_jpg/` and `photo_jpg/`. Set `BACKUP_DIR` to a **persistent** location that survives the lab reservation.

For a second experiment use a new `RUN_DIR`; never overwrite the baseline. Saving the notebook does not save checkpoints. With a local runtime, normal Colab `drive.mount()` is unavailable. Use the lab's supported persistent storage. With a Google-hosted runtime, mount Drive separately and set the backup path there.


In [ ]:
DATA_ROOT = PROJECT_ROOT/"task3_gan"/"data"
RUN_DIR = MEMBER                     # matches required member folder layout
BACKUP_DIR = None                    # REQUIRED before full training: Path("persistent destination")
CONFIG_NAME = "baseline"             # baseline or resizeconv
CODE_VERSION = "uncommitted"         # paste laptop commit SHA or a code-bundle label
RUN_ID = "pramod_baseline_seed42"     # unique label for team evidence collection
TRAIN_HOURS = 12.0                   # full learning-rate schedule, including later continuation
SESSION_HOURS = 4.0                  # this phase only; evaluate after four hours, then use 8.0
NUM_WORKERS = 2                      # use 0 if multiprocessing fails on the lab environment
BATCH_SIZE = 1
RUN_DIR = Path(RUN_DIR).expanduser().resolve()
DATA_ROOT = Path(DATA_ROOT).expanduser().resolve()
if BACKUP_DIR is not None:
    BACKUP_DIR = Path(BACKUP_DIR).expanduser().resolve()
SPLITS = RUN_DIR/"data_processed"/"splits.json"
for folder in ["monet_jpg", "photo_jpg"]:
    assert (DATA_ROOT/folder).is_dir(), f"Missing {DATA_ROOT/folder}"
print("Data:", DATA_ROOT, "Run:", RUN_DIR, "Backup:", BACKUP_DIR)


In [ ]:
def launch(script, *arguments):
    """Run in this kernel's Python environment; stream output and propagate failures."""
    command = [sys.executable, "-u", str(script), *map(str, arguments)]
    process = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                               text=True, bufsize=1, cwd=MEMBER)
    try:
        for line in process.stdout:
            print(line, end="", flush=True)
        status = process.wait()
    except KeyboardInterrupt:
        process.send_signal(signal.SIGINT)
        print("Stop requested. Trainer will finish a complete step and save; wait for it to exit.")
        process.wait()
        raise
    if status:
        raise RuntimeError(f"Command failed with exit code {status}; inspect the output above.")

launch(SRC/"preflight.py", "--data-root", DATA_ROOT)


## 4. Smoke test on CPU
Tests both architectures, cycle gradients, exact save/reload/resume against uninterrupted training, image export, metric mathematics and audit-sheet processing. This takes no training dataset and produces only temporary synthetic evidence.


In [ ]:
launch(SRC/"smoke_test.py")


## 5. Audit images and freeze splits
Run once. Reuse the same split manifest for every comparable experiment. Exact decoded-pixel duplicates are excluded before splitting; visually inspect possible near duplicates separately. Do not put official hidden-test images into the development folders.
The default 15% validation split supports the planned 15 examples per direction only if each domain has enough images.


In [ ]:
if SPLITS.exists():
    print("Reusing existing split manifest:", SPLITS)
else:
    launch(SRC/"data.py", "--data-root", DATA_ROOT, "--output", SPLITS, "--seed", 42, "--val-fraction", 0.15)
split_info = json.loads(SPLITS.read_text())
for d in ["A", "B"]:
    part = split_info["domains"][d]
    print(d, "train", len(part["train"]), "validation", len(part["val"]), "duplicates", len(part["duplicates"]))
    if len(part["val"]) < 15:
        raise RuntimeError("The planned audit needs 15 validation samples per domain. Decide a suitable split before training.")


## 6. Short GPU throughput benchmark
This is a separate disposable run, never the assignment baseline. It measures 30 updates using your actual model/data. The first few updates include warm-up effects. Do not compare its image quality or scores as if it were a trained model.


In [ ]:
import yaml, csv, statistics
assert torch.cuda.is_available(), "Select a CUDA-capable lab kernel before benchmarking."
base_config = yaml.safe_load((MEMBER/"configs"/(CONFIG_NAME+".yaml")).read_text())
base_config["train"].update(batch_size=BATCH_SIZE, num_workers=NUM_WORKERS)
BENCH_DIR = RUN_DIR/"data_processed"/("benchmark_"+CONFIG_NAME)
BENCH_DIR.mkdir(parents=True, exist_ok=True)
BENCH_CONFIG = BENCH_DIR/"benchmark.yaml"
if not (BENCH_DIR/"training_summary.json").exists():
    BENCH_CONFIG.write_text(yaml.safe_dump(base_config))
    launch(SRC/"train.py", "--config", BENCH_CONFIG, "--data-root", DATA_ROOT,
           "--splits", SPLITS, "--run-dir", BENCH_DIR, "--device", "cuda", "--stop-after", 30,
           "--code-version", CODE_VERSION)
else:
    assert yaml.safe_load(BENCH_CONFIG.read_text()) == base_config, "Benchmark config changed; choose a fresh benchmark directory."
    print("Reusing completed benchmark.")
history_file = sorted((BENCH_DIR/"reproducibility"/"raw_logs").glob("history_*.csv"))[-1]
with open(history_file) as f:
    benchmark_rows = list(csv.DictReader(f))
seconds_per_step = statistics.median(float(r["step_seconds"]) for r in benchmark_rows[-20:])
suggested_steps = max(2, int(TRAIN_HOURS*3600*0.85/seconds_per_step))
print("Median seconds/step:", seconds_per_step)
print("Suggested total steps including 15% overhead allowance:", suggested_steps)


## 7. Freeze the training configuration and verify backups
Review the displayed architecture, objective and estimated duration. The learning rate decays over the planned TOTAL steps. Resume must keep this exact config. Increasing steps after starting would change the learning-rate trajectory and is deliberately rejected.


In [ ]:
RUNTIME_CONFIG = RUN_DIR/"data_processed"/"runtime_config.yaml"
if RUNTIME_CONFIG.exists():
    train_config = yaml.safe_load(RUNTIME_CONFIG.read_text())
    print("Existing runtime configuration retained for exact resume.")
    if train_config["train"]["max_hours"] != TRAIN_HOURS + 0.5:
        raise RuntimeError("Existing schedule uses a different time budget. Use a fresh RUN_DIR for the overnight experiment, or restore TRAIN_HOURS to match the existing schedule for exact resume.")
else:
    train_config = base_config.copy()
    train_config["train"] = base_config["train"].copy()
    train_config["train"].update(max_steps=suggested_steps, decay_start=suggested_steps//2,
                                 max_hours=TRAIN_HOURS+0.5)
    RUNTIME_CONFIG.parent.mkdir(parents=True, exist_ok=True)
    RUNTIME_CONFIG.write_text(yaml.safe_dump(train_config))
print(yaml.safe_dump(train_config))
if BACKUP_DIR is None:
    raise RuntimeError("Set BACKUP_DIR to persistent storage in cell 3, then rerun this cell.")
assert BACKUP_DIR != RUN_DIR and RUN_DIR not in BACKUP_DIR.parents, "Backup must be outside the run directory to avoid recursive copying."
BACKUP_DIR.mkdir(parents=True, exist_ok=True)
probe = BACKUP_DIR/"task3_write_check.txt"
probe.write_text("Persistent backup write check")
assert probe.read_text() == "Persistent backup write check"
print("Backup directory is writable. Confirm it survives the lab session:", BACKUP_DIR)


## 8. Train four hours, evaluate, then resume
Set `START_TRAINING=True` after reviewing the benchmark and frozen configuration.
The default 12-hour budget is adjustable; it is not a claim about optimal convergence.
`BACKGROUND_TRAINING=True` launches a separate process with disk logs and checkpoints.
Do not also run foreground training for the same run directory.

Before leaving, check the status cell, confirm an actual checkpoint and backup exist,
and ensure the lab machine stays awake and your session will not be terminated.
Closing the notebook tab is not a shutdown request for this background process, but
lab policies, machine restarts, or process cleanup can still terminate it.
Notebook interruption does **not** stop a detached job.
Run sections 9 onward only after training has stopped; do not use Run All.

**First phase:** keep `TRAIN_HOURS=12.0` for the full schedule and launch with
`SESSION_HOURS=4.0`. It pauses after a complete optimizer step around four hours
and saves the entire training state. Wait for background status `finished`.
Run sections 9-12 to freeze and evaluate this checkpoint and make a fallback
submission CSV using the instructor evaluator. Preserve this snapshot and CSV.

**Continuation:** if the model is healthy, change `SESSION_HOURS` in this launch
cell to `8.0` and rerun it. The latest checkpoint is resumed automatically with
the same models, optimizer state, replay pools, EMA, RNG and full learning-rate
schedule. Do not change `TRAIN_HOURS`, architecture, losses or the frozen YAML.
Evaluate the new checkpoint again afterwards; retain the first-phase candidate.
A four-hour checkpoint is recoverable, but its quality still needs evaluation.


In [ ]:
START_TRAINING = False  # set True after reviewing cells 1-7
BACKGROUND_TRAINING = True
SESSION_HOURS = 4.0  # first phase; change to 8.0 AFTER evaluating to continue
if START_TRAINING:
    assert BACKUP_DIR is not None, "Persistent backup is required."
    command_args = ["--config", RUNTIME_CONFIG, "--data-root", DATA_ROOT,
                    "--splits", SPLITS, "--run-dir", RUN_DIR, "--device", "cuda",
                    "--backup-dir", BACKUP_DIR, "--code-version", CODE_VERSION,
                    "--session-hours", SESSION_HOURS]
    latest = RUN_DIR/"checkpoints"/"latest_checkpoint.pt"
    if latest.exists():
        command_args += ["--resume", latest]
    if BACKGROUND_TRAINING:
        launch(SRC/"background.py", "launch", "--run-dir", RUN_DIR, "--",
               sys.executable, "-u", str(SRC/"train.py"), *map(str, command_args))
    else:
        launch(SRC/"train.py", *command_args)
else:
    print("Training not started. Set START_TRAINING=True when ready.")


In [ ]:
# Rerun while training, and again when you return. This does not start a job.
launch(SRC/"background.py", "status", "--run-dir", RUN_DIR)
checkpoint = RUN_DIR/"checkpoints"/"latest_checkpoint.pt"
print("Latest checkpoint:", checkpoint, "exists:", checkpoint.exists())
if checkpoint.exists():
    print("Checkpoint modified:", time.ctime(checkpoint.stat().st_mtime))


## 9. Freeze an evaluation checkpoint
Choose a periodic checkpoint or copy the latest completed checkpoint. Compare candidate checkpoints and raw/EMA weights on the same validation split. Never select the model solely on low cycle loss. Full FID/KID and LPIPS require evaluation-weight downloads on first use.


In [ ]:
CANDIDATE_SOURCE = RUN_DIR/"checkpoints"/"latest_checkpoint.pt"
WEIGHTS = "ema"  # compare with "raw" using another export directory
assert CANDIDATE_SOURCE.exists(), "Complete a training segment first."
sys.path.insert(0, str(SRC))
from checkpointing import load_checkpoint, backup_file
from common import sha256
candidate = load_checkpoint(CANDIDATE_SOURCE)
step = candidate["step"]
SNAPSHOT = RUN_DIR/"checkpoints"/"evaluation"/f"step_{step:08d}.pt"
if SNAPSHOT.exists():
    assert sha256(SNAPSHOT) == sha256(CANDIDATE_SOURCE), "Conflicting snapshot; preserve both with distinct names."
else:
    backup_file(CANDIDATE_SOURCE, SNAPSHOT)
EXPORT_DIR = RUN_DIR/"outputs"/"evaluations"/f"step_{step:08d}_{WEIGHTS}"
if BACKUP_DIR is not None:
    backup_file(SNAPSHOT, BACKUP_DIR/"checkpoints"/"evaluation"/SNAPSHOT.name)
print("Snapshot:", SNAPSHOT, "Export:", EXPORT_DIR)
del candidate


In [ ]:
if not (EXPORT_DIR/"export_manifest.json").exists():
    launch(SRC/"infer.py", "--checkpoint", SNAPSHOT, "--data-root", DATA_ROOT,
           "--splits", SPLITS, "--output", EXPORT_DIR, "--weights", WEIGHTS,
           "--split", "val", "--device", "cuda")
else:
    print("Export exists; preserve it. Choose another directory for another candidate.")


In [ ]:
launch(MEMBER/"evaluate_local.py", "--export-dir", EXPORT_DIR, "--run-dir", RUN_DIR,
       "--device", "cuda", "--batch-size", 8)
launch(SRC/"report.py", "--run-dir", RUN_DIR, "--export-dir", EXPORT_DIR)
print((EXPORT_DIR/"full_metrics_report.csv").read_text())


## 10. Inspect and select
Review BOTH directions, sample counts, KID uncertainty, semantic preservation and failure grids. The local FID is not guaranteed to match Kaggle. Repeat cells 9 and evaluation for raw weights or another archived checkpoint if time allows.
Only after comparing candidates set `SELECT_THIS=True` and write a reason. Selection copies the entire fixed export and saves `best_model.pt`; it never chooses only attractive images.


In [ ]:
from IPython.display import display, Image as NotebookImage
preview_files = sorted((RUN_DIR/"outputs"/"plots").glob("preview_*.png"))
if preview_files:
    display(NotebookImage(filename=str(preview_files[-1]), width=768))
for filename in ["adversarial.png", "cycle_identity.png", "gradients.png", "learning_rate.png"]:
    path = RUN_DIR/"outputs"/"plots"/filename
    if path.exists(): display(NotebookImage(filename=str(path)))


In [ ]:
SELECT_THIS = False
SELECTION_REASON = ""  # actual validation comparison and visual findings
if SELECT_THIS:
    assert SELECTION_REASON.strip(), "Document why this candidate was selected."
    launch(SRC/"select_checkpoint.py", "--checkpoint", SNAPSHOT, "--export-dir", EXPORT_DIR,
           "--run-dir", RUN_DIR, "--reason", SELECTION_REASON)


## 11. Blinded human audit
Prepare only after choosing the model. Both raters receive the same packet with 30 fixed examples (15 per direction). Keep `private_key.json` private from raters. Each independently fills `rater_1.csv` or `rater_2.csv`. Higher artifact score means fewer artifacts.


In [ ]:
AUDIT_DIR = RUN_DIR/"outputs"/"human_audit"
if (RUN_DIR/"selection.json").exists() and not AUDIT_DIR.exists():
    selection = json.loads((RUN_DIR/"selection.json").read_text())
    assert selection["checkpoint_sha256"] == sha256(SNAPSHOT), "Audit must use the selected checkpoint."
    launch(SRC/"audit.py", "prepare", "--export-dir", EXPORT_DIR, "--output", AUDIT_DIR)
else:
    print("Select a checkpoint first, or reuse the already fixed audit packet.")


In [ ]:
RATINGS_COMPLETE = False
if RATINGS_COMPLETE:
    launch(SRC/"audit.py", "summarize", "--output", AUDIT_DIR)
    launch(SRC/"report.py", "--run-dir", RUN_DIR, "--export-dir", EXPORT_DIR)


## 12. Official Kaggle evaluator
The instructor evaluator is included as `src/evaluate_official.py`. It uses torchvision Inception (different from local diagnostic FID), the first 300 sorted files per folder, matching counts, and index-wise cosine distance. It writes `ID,FID,MiFID`. Do not rearrange or hand-pick outputs to improve the score. Keep fixed held-out metrics separately for the report.

After freezing and evaluating a candidate on validation, create a separate official
export. Use this only for the supplied first-300 scoring protocol and the approved
development source set; follow any separate instructor input-set requirements.
The deterministic prefix preserves the first 300 outputs of a full all-input export.
Full local validation still uses all validation samples and cycle images.

Run in a new cell after training has stopped:
```python
OFFICIAL_DIR = RUN_DIR/"outputs"/"official"/f"{EXPORT_DIR.name}_first300"
if not (OFFICIAL_DIR/"export_manifest.json").exists():
    launch(SRC/"infer.py", "--checkpoint", SNAPSHOT, "--data-root", DATA_ROOT,
           "--splits", SPLITS, "--output", OFFICIAL_DIR, "--weights", WEIGHTS,
           "--split", "all", "--max-per-domain", 300, "--predictions-only",
           "--device", "cuda")
OFFICIAL_CSV = RUN_DIR/f"submission_{EXPORT_DIR.name}.csv"
if not OFFICIAL_CSV.exists():
    launch(SRC/"evaluate_official.py", "--data-root", DATA_ROOT,
           "--export-dir", OFFICIAL_DIR, "--output", OFFICIAL_CSV, "--batch-size", 16)
print(OFFICIAL_CSV.read_text())
```
Each checkpoint gets a distinct CSV so the four-hour fallback stays intact.
Copy your chosen candidate CSV to `submission.csv` for the final repository.

The reference folders follow the instructor script and include training images; this official-protocol score is not held-out generalization evidence. Confirm any separate competition source-image requirements. No automatic upload is performed.

## 13. Collect evidence and back up the full run
Run after training stops. Save the executed notebook separately too.


In [ ]:
launch(SRC/"collect_evidence.py", "--run-dir", RUN_DIR, "--repo-root", PROJECT_ROOT, "--run-id", RUN_ID)
if BACKUP_DIR is not None:
    # Explicit include list avoids recursively copying the backup into itself.
    for name in ["checkpoints", "outputs", "reproducibility", "data_processed"]:
        source = RUN_DIR/name
        if source.exists(): shutil.copytree(source, BACKUP_DIR/name, dirs_exist_ok=True)
    for name in ["training_summary.json", "selection.json", "full_metrics_report.csv",
                 "metrics_report.csv", "results_draft.md", "submission.csv", "submission_provenance.json"]:
        source = RUN_DIR/name
        if source.exists(): shutil.copy2(source, BACKUP_DIR/name)
    latest = RUN_DIR/"checkpoints"/"latest_checkpoint.pt"
    if latest.exists():
        assert sha256(latest) == sha256(BACKUP_DIR/"checkpoints"/latest.name)
    print("Backup complete; latest checkpoint checksum verified. Save the executed notebook separately.")


## 14. Laptop commits and report
Bring back the executed notebook, source changes (if any), raw logs, manifests, metrics, plots and analysis. Keep large binaries in persistent storage with checkpoint hashes and retrieval instructions. Commit code before runs and results after runs; do not manufacture history.

Complete `results.md` and `failure_analysis.md` from real evidence. The generated `results_draft.md` is a scaffold, not finished scientific analysis. Coordinate evaluation settings with your teammate. Explain architecture, objectives, metric limitations and observed failures in the viva.


## 15. Quick official score for the chosen frozen snapshot
After training stops, rerun Section 9's **first cell** to update `SNAPSHOT`. This cell uses that snapshot, not the active checkpoint. It reuses matching official results or generates only 300 JPGs per direction and computes the instructor FID/MiFID and expected displayed Kaggle score. It does not replace the full local metrics or human audit.


In [ ]:
import runpy
quick_result = runpy.run_path(str(SRC/"quick_kaggle_score.py"),
    init_globals={"RUN_DIR": RUN_DIR, "DATA_ROOT": DATA_ROOT, "SNAPSHOT": SNAPSHOT, "WEIGHTS": WEIGHTS})
OFFICIAL_CSV = Path(quick_result["OFFICIAL_CSV"])


## 16. Downloadable completed-snapshot package
Run after the selected training segment has finished and Section 9's first cell has frozen it. Set `PACKAGE_STEP=None` to select the highest completed step, or set it to the printed step to archive that exact snapshot. The ZIP includes completed matching training-history segments, exact score CSVs, generated images, source notebooks and the frozen checkpoint. The manifest lists missing rubric evidence. The dataset is excluded; restore the same original dataset separately. No upload or email occurs.

Download the resulting file from `task3_gan/transfer/`. An executed notebook outside the member folder can be included by setting `EXECUTED_NOTEBOOK` to its saved path. The starter file in source is a clean runnable notebook, not a substitute for the executed notebook.


In [ ]:
PACKAGE_STEP = None  # default: latest completed phase; or enter the exact snapshot step
EXECUTED_NOTEBOOK = None  # optional Path to the SAVED executed notebook
package_args = ["--run-dir", RUN_DIR, "--full", "--weights", WEIGHTS]
if PACKAGE_STEP is not None:
    package_args += ["--step", PACKAGE_STEP]
if EXECUTED_NOTEBOOK is not None:
    package_args += ["--notebook", EXECUTED_NOTEBOOK]
launch(SRC/"package_completed_run.py", *package_args)
